# FedGTP under V2X communication impairment

**Research question.** How do realistic V2X communication constraints affect *distributed*
traffic forecasting accuracy?

This notebook runs the **official FedGTP** (Yang et al., KDD 2024) — cloned at a pinned SHA and
never edited — with a thin transport wrapper underneath that injects latency and missed
deadlines, and counts every byte the federation actually sends.

| | |
|---|---|
| official repo | `LarryHawkingYoung/KDD2024_FedGTP` |
| pinned SHA | `ff128f5bed7780b384776d03fe0bad8110575a64` |
| dataset | PeMSD7 (ships inside the repo — the authors' own preprocessing) |
| adapter | `pipeline/federated/v2x_impairment.py` (wraps send/recv only) |

**These impairments are SIMULATED.** The `mosaic_v2x` latency distribution comes from Eclipse
MOSAIC's ITS-G5 model — an official simulator — not from radio hardware. Nothing here is a
measurement of a real V2X deployment.

Needs a GPU runtime: upstream's `client.py` calls `get_memory_usage('cuda')` unconditionally.
*Runtime → Change runtime type → GPU.*


In [ ]:
!nvidia-smi -L
import torch; print('cuda available:', torch.cuda.is_available())


## 1. Repo and the pinned upstream checkout

The pin is verified before anything runs — if the SHA does not match, the launcher refuses.


In [ ]:
import pathlib, subprocess, os, json
REPO = pathlib.Path('/content/thesis')
if not REPO.exists():
    !git clone -q --branch rebuild-120h https://github.com/auth889-ai/thesis.git {REPO}
os.chdir(REPO)
!git log --oneline -1


In [ ]:
WS = pathlib.Path('/root/.dhakaflow/upstream'); WS.mkdir(parents=True, exist_ok=True)
os.environ['DHAKAFLOW_WORKSPACE'] = str(WS)
!python pipeline/upstream/fetch_upstream.py --name FedGTP
!git -C {WS}/FedGTP rev-parse HEAD


## 2. Dependencies

The authors' `install.sh` targets conda; on Colab the same packages come from pip. Torch is
already present in the Colab image.


In [ ]:
!pip -q install networkx scikit-learn tqdm pandas


## 3. Sanity: does the wrapper behave over real sockets?

Run the adapter's own tests before spending GPU time on a sweep.


In [ ]:
!python -m pytest tests/federated/test_v2x_impairment.py -q


## 4. Baseline reproduction — the `ideal` profile

Upstream, unimpaired. This is the number every impaired run is compared against, and it is
what decides `REPRODUCTION_PASS` in `results/upstream/REPRODUCTIONS.md`.

Start small (`--epochs 5`) to confirm the pipeline runs end to end, then raise to the authors'
default of 200.


In [ ]:
!python pipeline/federated/run_fedgtp.py --profile ideal --epochs 5 --device cuda:0


## 5. The impairment sweep

Conditions **A** ideal, **B** the simulated MOSAIC V2X profile, **C** latency stress,
**D** packet-loss stress. Each run records forecast MAE/RMSE/MAPE (parsed from upstream's own
evaluation output), communication bytes, messages, rounds, wall time, injected delay and the
drops that actually occurred.

Raise `--epochs` to 200 for the reportable run; 200 epochs x 8 profiles is long, so consider
running the sweep in two halves.


In [ ]:
!python pipeline/federated/run_fedgtp.py --sweep --epochs 50 --device cuda:0


## 6. Results back to Drive

Nothing is trusted until it is written to a file with its provenance attached.


In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import shutil
OUT = pathlib.Path('/content/drive/MyDrive/dhakaflow/federated'); OUT.mkdir(parents=True, exist_ok=True)
for f in (REPO/'results/federated').rglob('*.json'):
    dst = OUT/f.relative_to(REPO/'results/federated')
    dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(f, dst)
print('copied to', OUT)
print(json.dumps(json.loads((REPO/'results/federated/fedgtp_impairment_PeMSD7.json').read_text())['runs'][0], indent=1)[:800])


## 7. Pull them onto the Mac

```bash
rclone copy gdrive2tb:dhakaflow/federated results/federated
python pipeline/federated/summarise_impairment.py
```
